# 02. Preprocessing

This notebook translates the modeling decisions from the EDA into a reproducible preprocessing workflow.

The main goals are to:

- define the modeling feature sets,
- separate numeric and categorical preprocessing,
- handle categorical encoding safely,
- decide whether numeric scaling is needed,
- design an appropriate representation for `pdays`,
- compare Core and Extended feature sets,
- and build the preprocessing logic using scikit-learn `Pipeline` and `ColumnTransformer`.

All preprocessing steps that learn from the data will be fitted on the training data only.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load the data

The raw Bank Marketing dataset is loaded directly from the project data directory.

The original dataframe is kept intact. In particular, `duration` is not removed from the raw data; it will be excluded only from the modeling feature sets because it is unavailable at the intended prediction time.

In [3]:
PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

PosixPath('/home/moonk/projects/bank-marketing-ml')

In [4]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

RAW_DATA_PATH

PosixPath('/home/moonk/projects/bank-marketing-ml/data/raw/bank-additional/bank-additional-full.csv')

In [5]:
df = pd.read_csv(RAW_DATA_PATH, sep=";")

In [6]:
df.shape

(41188, 21)

In [7]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Define the modeling target and exclusions

The target variable is `y`, where:

- `yes` means the client subscribed to a term deposit,
- `no` means the client did not subscribe.

The intended prediction time is immediately before a planned marketing contact.

`duration` records the duration of the current call and is therefore not available at prediction time. It is treated as a leakage feature and excluded from all modeling feature sets.

In [8]:
target = "y"
excluded_features = ["duration"]

After excluding the target and `duration`, 19 candidate predictors remain for modeling.

In [9]:
modeling_features = [
    col
    for col in df.columns
    if col not in [target, *excluded_features]
]

len(modeling_features)

19

In [10]:
modeling_features

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed']

## 3. train / validation / test split

The dataset has temporal structure in its row order, and the target prevalence and macroeconomic variables change substantially over time.

Therefore, we use a chronological split rather than a random or stratified split:

- train: first 70%
- validation: next 20%
- test: final 10%

The training set will be used to fit preprocessing steps and models.

The validation set will be used for model comparison and preprocessing decisions.

The test period was already inspected at an aggregate level during EDA. From this point onward, it is frozen for final evaluation.

In [11]:
n_rows = len(df)

train_end = int(n_rows * 0.7)
valid_end = int(n_rows * 0.9)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

In [12]:
train_df.shape, valid_df.shape, test_df.shape

((28831, 21), (8238, 21), (4119, 21))

## 4. Separate features and target

The target is converted to a binary representation:

- `no` → 0
- `yes` → 1

`duration` is excluded only from the modeling features and remains available in the original split dataframes.

In [13]:
X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()
X_test = test_df[modeling_features].copy()

In [14]:
y_train = train_df[target].map({"no": 0, "yes": 1})
y_valid = valid_df[target].map({"no": 0, "yes": 1})
y_test = test_df[target].map({"no": 0, "yes": 1})

In [15]:
X_train.shape, X_valid.shape, X_test.shape

((28831, 19), (8238, 19), (4119, 19))

In [16]:
y_train.value_counts(normalize=True)

y
0    0.944296
1    0.055704
Name: proportion, dtype: float64

## 5. Define preprocessing feature groups

The modeling features contain both numeric and categorical variables, but preprocessing groups should not be determined by pandas dtype alone.

In particular, `pdays` is stored as a numeric variable, but the value `999` has special meaning and dominates the training data. Therefore, it will be handled separately from ordinary numeric features.

We define three preprocessing groups:

- ordinary numeric features,
- categorical features,
- special features requiring separate treatment.

In [17]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

special_features = [
    "pdays",
]

In [18]:
len(numeric_features), len(categorical_features), len(special_features)

(8, 10, 1)

### Why is `pdays` separated?

`pdays` represents the number of days since a previous contact, but `999` is used as a special sentinel-like value.

In the training data, almost all observations have `pdays = 999`, and this value cannot be interpreted simply as an ordinary continuous magnitude.

Therefore, `pdays` will not automatically follow the same preprocessing path as the other numeric features. Its representation will be evaluated separately.

## 6. Categorical encoding

The categorical features have relatively low cardinality, so one-hot encoding is a natural baseline.

For the initial preprocessing design:

- `"unknown"` is kept as an explicit category,
- rare categories are not manually grouped,
- no category is dropped,
- categories unseen during training are handled safely.

Because the validation and test periods occur later in time, they may contain categories that were not observed in the training period. Therefore, the encoder should not fail when an unseen category appears.

In [19]:
from sklearn.preprocessing import OneHotEncoder

In [20]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore"
)

### Why `handle_unknown="ignore"`?

The encoder learns its categories from the training data only.

Since the validation and test sets represent later time periods, a categorical level that was absent from training may appear later.

With `handle_unknown="ignore"`, such categories can still be transformed without raising an error. This preserves the train-only fitting rule while allowing the preprocessing pipeline to handle future data safely.

In [ ]:
categorical_encoder.fit(
    X_train[categorical_features]
)

,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infrequen

In [22]:
for feature, categories in zip(
    categorical_features,
    categorical_encoder.categories_,
):
    print(feature, ":", categories)

job : ['admin.' 'blue-collar' 'entrepreneur' 'housemaid' 'management' 'retired'
 'self-employed' 'services' 'student' 'technician' 'unemployed' 'unknown']
marital : ['divorced' 'married' 'single' 'unknown']
education : ['basic.4y' 'basic.6y' 'basic.9y' 'high.school' 'illiterate'
 'professional.course' 'university.degree' 'unknown']
default : ['no' 'unknown' 'yes']
housing : ['no' 'unknown' 'yes']
loan : ['no' 'unknown' 'yes']
contact : ['cellular' 'telephone']
month : ['apr' 'aug' 'dec' 'jul' 'jun' 'mar' 'may' 'nov' 'oct']
day_of_week : ['fri' 'mon' 'thu' 'tue' 'wed']
poutcome : ['failure' 'nonexistent' 'success']


In [25]:
X_train_cat = categorical_encoder.transform(
    X_train[categorical_features]
)

X_train_cat.shape

(28831, 52)

### Keep the sparse representation

One-hot encoding creates many zero values.

`OneHotEncoder` returns a sparse representation by default, which is suitable for the initial pipeline and avoids unnecessarily converting the encoded data into a dense matrix.

We therefore keep the default sparse output for now.

In [26]:
encoded_feature_names = (
    categorical_encoder.get_feature_names_out(
        categorical_features
    )
)

len(encoded_feature_names)

52

### Why not drop a category?

For the initial baseline, all one-hot encoded categories are retained.

Although dropping one category can remove exact linear dependence among dummy variables, the baseline models may use regularization, and keeping the full representation makes the encoding symmetric and easier to inspect.

We can revisit this choice later if it becomes relevant for a specific linear model.

## 7. Numeric scaling

Numeric features are measured on very different scales.

For example:

- `age` is typically in the tens,
- `campaign` is a small count,
- `euribor3m` is a rate,
- `nr.employed` is around several thousand.

This matters for models such as logistic regression because the optimization and regularization operate on the numeric feature values directly.

Tree-based models, however, generally do not require feature scaling.

Therefore, the initial design will use scaling for linear baseline models, while tree-based baselines can later use a simpler numeric branch.

In [30]:
from sklearn.preprocessing import StandardScaler

In [31]:
numeric_scaler = StandardScaler()

### Why use `StandardScaler` for the linear baseline?

The initial linear baseline will likely use logistic regression.

Scaling is useful here because:

- numeric features have different units and ranges,
- regularization acts more consistently when features are on comparable scales,
- optimization is generally better behaved.

The scaler must be fitted on the training data only.

In [33]:
numeric_scaler.fit(
    X_train[numeric_features]
)

StandardScaler()

In [36]:
X_train_num = numeric_scaler.transform(
    X_train[numeric_features]
)

X_train_num.shape

(28831, 8)

### Scaling strategy by model family

We will not assume that every model needs the same numeric preprocessing.

For linear models:

- numeric features will be standardized.

For tree-based models:

- numeric features can usually be passed through without scaling.

The categorical encoding strategy can remain the same initially.

### No additional numeric transformation yet

Some numeric features, especially `campaign`, are right-skewed.

For the initial baseline, we do not apply:

- log transformation,
- clipping,
- winsorization,
- manual outlier removal.

The goal is to establish a simple and reproducible baseline first. Additional transformations can later be tested against validation performance.

## 8. Special handling for `pdays`

`pdays` requires separate treatment because the value `999` dominates the training data and should not be interpreted as an ordinary continuous magnitude.

A value of `999` is related to previous campaign contact history, but the EDA showed that it does not perfectly correspond to `previous = 0`.

Therefore, we should avoid simply treating `999` as "999 days ago".

For the initial comparison, we will consider two representations:

1. **Raw representation**
   - keep `pdays` as it is.

2. **Sentinel-aware representation**
   - `pdays_is_999`: whether the special value `999` is present,
   - `pdays_recency`: retain the observed day count when `pdays != 999`, otherwise use `0`.

The indicator allows the model to distinguish the sentinel case from an actual recency value.

In [39]:
X_train["pdays"].value_counts().sort_index()

pdays
0          2
1          4
2          1
3          7
4          8
5         12
6         10
7          1
8          1
9          4
10         2
11         1
999    28778
Name: count, dtype: int64

In [40]:
(X_train["pdays"] == 999).mean()

np.float64(0.9981617009468975)

In [41]:
pdays_train = pd.DataFrame(
    {
        "pdays_is_999": (X_train["pdays"] == 999).astype(int),
        "pdays_recency": X_train["pdays"].where(
            X_train["pdays"] != 999,
            0,
        ),
    },
    index=X_train.index,
)

pdays_train.head()

,pdays_is_999,pdays_recency
0,1,0
1,1,0
2,1,0
3,1,0
4,1,0


### Why use `0` for the sentinel case in `pdays_recency`?

The value `0` is only a placeholder for the recency component when `pdays = 999`.

It is not intended to mean that the previous contact occurred zero days ago.

The separate `pdays_is_999` indicator tells the model whether the recency value is applicable, so the two features should be interpreted together.

In [42]:
pdays_train.describe()

,pdays_is_999,pdays_recency
count,28831.000000,28831.000000
mean,0.998162,0.009053
std,0.042837,0.236655
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,1.000000,0.000000
max,1.000000,11.000000


### Initial `pdays` comparison

We will not decide yet that the engineered representation is better.

The baseline comparison will be:

- **Raw**: use the original `pdays`,
- **Sentinel-aware**: replace the original `pdays` with `pdays_is_999` and `pdays_recency`.

Both representations will be evaluated on the same chronological validation set.

Because `pdays`, `previous`, and `poutcome` describe related aspects of previous campaign history, we keep `previous` and `poutcome` in the model and let validation determine whether the additional `pdays` representation is useful.

## 9. Build preprocessing configurations

We now combine the preprocessing decisions into reusable scikit-learn `ColumnTransformer` objects.

Two preprocessing configurations will be compared for `pdays`:

1. **Raw `pdays` baseline**
   - treat the original `pdays` value as an ordinary numeric feature.

2. **Sentinel-aware `pdays`**
   - `pdays_is_999`: indicates whether `pdays` is equal to `999`,
   - `pdays_recency`: keeps the observed day count when `pdays != 999` and uses `0` otherwise.

For both configurations:

- ordinary numeric features are standardized with `StandardScaler`,
- categorical features are encoded with `OneHotEncoder`,
- unseen categorical levels are handled safely with `handle_unknown="ignore"`,
- preprocessing is fitted on the training data only.

The two `pdays` representations will later be compared using the same baseline model and chronological validation set.

In [67]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

### 9.1 Raw `pdays` baseline

The first configuration provides a simple reference point.

In this version, `pdays` is treated as an ordinary numeric feature and follows the same preprocessing path as the other numeric variables.

In [70]:
raw_numeric_features = numeric_features + ["pdays"]

raw_numeric_features

['age',
 'campaign',
 'previous',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'pdays']

In [71]:
preprocessor_raw_pdays = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            raw_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

The `ColumnTransformer` applies different preprocessing steps to different column groups:

- numeric features, including raw `pdays`, are passed through `StandardScaler`,
- categorical features are passed through `OneHotEncoder`,
- columns not explicitly included in the transformer are dropped.

At this point, the preprocessor has only been defined. It has not learned anything from the data yet.

In [72]:
X_train_raw_prepared = preprocessor_raw_pdays.fit_transform(
    X_train
)

In [73]:
X_valid_raw_prepared = preprocessor_raw_pdays.transform(
    X_valid
)

The preprocessing parameters are learned from the training data only.

For example:

- `StandardScaler` learns the training means and standard deviations,
- `OneHotEncoder` learns the categorical levels observed in training.

The validation data is only transformed using these already-fitted preprocessing rules.

In [74]:
X_train_raw_prepared.shape, X_valid_raw_prepared.shape

((28831, 61), (8238, 61))

In [75]:
raw_feature_names = (
    preprocessor_raw_pdays
    .get_feature_names_out()
)

len(raw_feature_names)

61

In [76]:
raw_feature_names

array(['num__age', 'num__campaign', 'num__previous', 'num__emp.var.rate',
       'num__cons.price.idx', 'num__cons.conf.idx', 'num__euribor3m',
       'num__nr.employed', 'num__pdays', 'cat__job_admin.',
       'cat__job_blue-collar', 'cat__job_entrepreneur',
       'cat__job_housemaid', 'cat__job_management', 'cat__job_retired',
       'cat__job_self-employed', 'cat__job_services', 'cat__job_student',
       'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown',
       'cat__marital_divorced', 'cat__marital_married',
       'cat__marital_single', 'cat__marital_unknown',
       'cat__education_basic.4y', 'cat__education_basic.6y',
       'cat__education_basic.9y', 'cat__education_high.school',
       'cat__education_illiterate', 'cat__education_professional.course',
       'cat__education_university.degree', 'cat__education_unknown',
       'cat__default_no', 'cat__default_unknown', 'cat__default_yes',
       'cat__housing_no', 'cat__housing_unknown', 'cat__housing_yes',
   

### 9.2 Sentinel-aware `pdays`

The second configuration represents `pdays` using two separate pieces of information:

- `pdays_is_999`: whether the special value `999` is present,
- `pdays_recency`: the observed number of days when `pdays != 999`, and `0` otherwise.

Unlike the raw baseline, this representation does not treat `999` as an ordinary numeric magnitude.

`pdays_recency` is standardized like the other numeric features, while `pdays_is_999` is kept as a binary indicator.

The transformation is implemented inside the preprocessing pipeline so that it can be applied automatically to validation, test, and future unseen data.

In [77]:
from sklearn.pipeline import Pipeline

In [78]:
def transform_pdays_recency(X):
    pdays = X["pdays"]

    return pd.DataFrame(
        {
            "pdays_recency": pdays.where(
                pdays != 999,
                0,
            )
        },
        index=X.index,
    )


def transform_pdays_indicator(X):
    pdays = X["pdays"]

    return pd.DataFrame(
        {
            "pdays_is_999": (pdays == 999).astype(int)
        },
        index=X.index,
    )

In [79]:
pdays_recency_transformer = FunctionTransformer(
    transform_pdays_recency,
    feature_names_out=lambda transformer, input_features: [
        "pdays_recency"
    ],
)

pdays_indicator_transformer = FunctionTransformer(
    transform_pdays_indicator,
    feature_names_out=lambda transformer, input_features: [
        "pdays_is_999"
    ],
)

In [80]:
pdays_recency_pipeline = Pipeline(
    steps=[
        ("transform", pdays_recency_transformer),
        ("scale", StandardScaler()),
    ]
)

In [81]:
preprocessor_sentinel_pdays = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "pdays_recency",
            pdays_recency_pipeline,
            ["pdays"],
        ),
        (
            "pdays_indicator",
            pdays_indicator_transformer,
            ["pdays"],
        ),
    ],
    remainder="drop",
)

In [82]:
X_train_sentinel_prepared = (
    preprocessor_sentinel_pdays.fit_transform(X_train)
)

In [83]:
X_valid_sentinel_prepared = (
    preprocessor_sentinel_pdays.transform(X_valid)
)

In [84]:
(
    X_train_raw_prepared.shape,
    X_train_sentinel_prepared.shape,
)

((28831, 61), (28831, 62))

In [85]:
sentinel_feature_names = (
    preprocessor_sentinel_pdays
    .get_feature_names_out()
)

sentinel_feature_names

array(['num__age', 'num__campaign', 'num__previous', 'num__emp.var.rate',
       'num__cons.price.idx', 'num__cons.conf.idx', 'num__euribor3m',
       'num__nr.employed', 'cat__job_admin.', 'cat__job_blue-collar',
       'cat__job_entrepreneur', 'cat__job_housemaid',
       'cat__job_management', 'cat__job_retired',
       'cat__job_self-employed', 'cat__job_services', 'cat__job_student',
       'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown',
       'cat__marital_divorced', 'cat__marital_married',
       'cat__marital_single', 'cat__marital_unknown',
       'cat__education_basic.4y', 'cat__education_basic.6y',
       'cat__education_basic.9y', 'cat__education_high.school',
       'cat__education_illiterate', 'cat__education_professional.course',
       'cat__education_university.degree', 'cat__education_unknown',
       'cat__default_no', 'cat__default_unknown', 'cat__default_yes',
       'cat__housing_no', 'cat__housing_unknown', 'cat__housing_yes',
       'cat__loan

### Preprocessing configurations

We now have two reusable preprocessing configurations.

**Raw `pdays` baseline**

- numeric features → `StandardScaler`
- categorical features → `OneHotEncoder`
- `pdays` → raw numeric value

**Sentinel-aware `pdays`**

- numeric features → `StandardScaler`
- categorical features → `OneHotEncoder`
- `pdays_recency` → `StandardScaler`
- `pdays_is_999` → binary indicator

Both preprocessors are fitted on the training data only and can be applied directly to unseen data.

We do not choose between the two representations yet. Their usefulness will be compared using the same baseline model and chronological validation set.